In [ ]:
{
    "cells": [
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "# ADUserReview.ipynb\n",
                "### Purpose: Review AD Group and GPO data and tag decisions interactively using dropdowns and widgets."
            ]
        },
        {
            "cell_type": "code",
            "execution_count": null,
            "metadata": {},
            "outputs": [],
            "source": [
                "# === Setup & Imports ===\n",
                "from pathlib import Path\n",
                "import json\n",
                "import pandas as pd\n",
                "import ipywidgets as widgets\n",
                "from IPython.display import display, Markdown, clear_output\n",
                "\n",
                "exports_dir = Path(\"../exports\")\n",
                "user_dirs = [d.name for d in exports_dir.iterdir() if d.is_dir()]\n",
                "\n",
                "user_picker = widgets.Dropdown(options=user_dirs, description=\"Select User:\")\n",
                "group_box = widgets.VBox([])\n",
                "gpo_box = widgets.VBox([])\n",
                "group_widgets = {}\n",
                "gpo_widgets = {}\n",
                "json_path = None\n",
                "df_groups = None\n",
                "df_gpos = None\n",
                "current_user_dir = None\n",
                "\n",
                "display(Markdown(\"## Select AD Export Folder\"))\n",
                "display(user_picker)\n",
                "display(group_box)\n",
                "display(gpo_box)"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": null,
            "metadata": {},
            "outputs": [],
            "source": [
                "# === Load latest JSON ===\n",
                "def load_latest_json(user):\n",
                "    user_path = exports_dir / user\n",
                "    json_files = sorted(user_path.glob(\"ad_user_summary_*.json\"), reverse=True)\n",
                "    if not json_files:\n",
                "        raise FileNotFoundError(f\"No JSON files found in {user_path}\")\n",
                "    return json_files[0], user_path"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": null,
            "metadata": {},
            "outputs": [],
            "source": [
                "# === On Folder Select ===\n",
                "def on_user_change(change):\n",
                "    clear_output(wait=True)\n",
                "    display(Markdown(\"## Select AD Export Folder\"))\n",
                "    display(user_picker)\n",
                "    display(group_box)\n",
                "    display(gpo_box)\n",
                "\n",
                "    global df_groups, df_gpos, json_path, current_user_dir, group_widgets, gpo_widgets\n",
                "    group_widgets.clear()\n",
                "    gpo_widgets.clear()\n",
                "\n",
                "    json_path, current_user_dir = load_latest_json(change[\"new\"])\n",
                "    print(f\"[INFO] Loaded: {json_path.name}\")\n",
                "\n",
                "    with open(json_path, \"r\", encoding=\"utf-8\") as f:\n",
                "        ad_data = json.load(f)\n",
                "\n",
                "    df_groups = pd.DataFrame(ad_data.get(\"Groups\", []), columns=[\"GroupName\"])\n",
                "    df_groups[\"Decision\"] = \"\"\n",
                "\n",
                "    group_items = []\n",
                "    for group in df_groups[\"GroupName\"]:\n",
                "        dropdown = widgets.Dropdown(\n",
                "            options=[\"Keep\", \"Remove\", \"CAB Review\", \"Unsure\"],\n",
                "            value=\"Unsure\",\n",
                "            description=group[:30] + (\"...\" if len(group) > 30 else \"\"),\n",
                "            style={'description_width': 'initial'},\n",
                "            layout=widgets.Layout(width='80%')\n",
                "        )\n",
                "        group_widgets[group] = dropdown\n",
                "        group_items.append(dropdown)\n",
                "    group_box.children = [Markdown(\"### Group Membership Review\")] + group_items\n",
                "\n",
                "    df_gpos = pd.DataFrame(ad_data.get(\"GPOs\", []), columns=[\"GPOName\"])\n",
                "    df_gpos[\"Decision\"] = \"\"\n",
                "\n",
                "    gpo_items = []\n",
                "    for gpo in df_gpos[\"GPOName\"]:\n",
                "        dropdown = widgets.Dropdown(\n",
                "            options=[\"Keep\", \"Remove\", \"CAB Review\", \"Unsure\"],\n",
                "            value=\"Unsure\",\n",
                "            description=gpo[:30] + (\"...\" if len(gpo) > 30 else \"\"),\n",
                "            style={'description_width': 'initial'},\n",
                "            layout=widgets.Layout(width='80%')\n",
                "        )\n",
                "        gpo_widgets[gpo] = dropdown\n",
                "        gpo_items.append(dropdown)\n",
                "    gpo_box.children = [Markdown(\"### GPO Linkage Review\")] + gpo_items\n",
                "\n",
                "user_picker.observe(on_user_change, names=\"value\")"
            ]
        },
        {
            "cell_type": "code",
            "execution_count": null,
            "metadata": {},
            "outputs": [],
            "source": [
                "# === Save + Export Buttons ===\n",
                "def save_group_decisions(b):\n",
                "    for group, dropdown in group_widgets.items():\n",
                "        df_groups.loc[df_groups[\"GroupName\"] == group, \"Decision\"] = dropdown.value\n",
                "    path = current_user_dir / \"interactive_decision_groups.csv\"\n",
                "    df_groups.to_csv(path, index=False)\n",
                "    print(f\"[INFO] Group decisions saved: {path}\")\n",
                "\n",
                "def save_gpo_decisions(b):\n",
                "    for gpo, dropdown in gpo_widgets.items():\n",
                "        df_gpos.loc[df_gpos[\"GPOName\"] == gpo, \"Decision\"] = dropdown.value\n",
                "    path = current_user_dir / \"interactive_decision_gpos.csv\"\n",
                "    df_gpos.to_csv(path, index=False)\n",
                "    print(f\"[INFO] GPO decisions saved: {path}\")\n",
                "\n",
                "def export_markdown_summary(b):\n",
                "    lines = [\"# AD Review Summary\\n\"]\n",
                "    lines.append(\"## Group Membership Decisions\\n\")\n",
                "    for _, row in df_groups.iterrows():\n",
                "        lines.append(f\"- **{row['GroupName']}**: {row['Decision']}\")\n",
                "    lines.append(\"\\n## GPO Linkage Decisions\\n\")\n",
                "    for _, row in df_gpos.iterrows():\n",
                "        lines.append(f\"- **{row['GPOName']}**: {row['Decision']}\")\n",
                "    path = current_user_dir / \"decision_summary.md\"\n",
                "    with open(path, \"w\", encoding=\"utf-8\") as f:\n",
                "        f.write(\"\\n\".join(lines))\n",
                "    print(f\"[INFO] Markdown summary saved: {path}\")\n",
                "\n",
                "display(widgets.Button(description=\"\ud83d\udcbe Save Group Decisions\", on_click=save_group_decisions))\n",
                "display(widgets.Button(description=\"\ud83d\udcbe Save GPO Decisions\", on_click=save_gpo_decisions))\n",
                "display(widgets.Button(description=\"\ud83d\udcdd Export Markdown Summary\", on_click=export_markdown_summary))"
            ]
        }
    ],
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3"
        },
        "language_info": {
            "name": "python",
            "version": "3.11"
        }
    },
    "nbformat": 4,
    "nbformat_minor": 5
}
